# ForgeGPT: full training run on a Kaggle T4

Trains the ~14M-parameter model (6 layers, 6 heads, width 384, context 256) on all
149M training tokens, then runs three short ablations.

**Before running**
1. Settings → Accelerator → **GPU T4 x2** (one GPU is used) and Internet **on**.
2. Add your data as a Kaggle dataset named `forgegpt-tokens-v1` containing
   `train.bin`, `val.bin`, `test.bin`, `meta.json` (from `data/tokens/v1/`) and
   `bpe-8k.json` (from `data/tokenizer/v1/`). About 330 MB.
3. Set `REPO_URL` below to your GitHub repo.

**Kaggle sessions time out.** Training saves `ckpt_latest.pt` at every evaluation.
To resume in a new session, add this notebook's previous output as an input and run
the "Resume" cell before training.

In [ ]:
REPO_URL = "https://github.com/<your-username>/forgegpt.git"  # <- change this
!git clone -q $REPO_URL forgegpt && cd forgegpt && pip install -q -e .
import sys

sys.path.insert(0, "/kaggle/working/forgegpt/src")
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Resume (optional): copy the last checkpoint from a previous version's output

In [ ]:
import shutil, pathlib

prev = pathlib.Path("/kaggle/input/forgegpt-train-output/runs/t4_6L384/ckpt_latest.pt")  # adjust
dst = pathlib.Path("/kaggle/working/runs/t4_6L384")
dst.mkdir(parents=True, exist_ok=True)
if prev.exists():
    shutil.copy(prev, dst / "ckpt_latest.pt")
    print("will resume from", prev)
else:
    print("no previous checkpoint, starting fresh")

## Main run (~45–60 min on a T4)

In [ ]:
%cd /kaggle/working/forgegpt
!python -m forgegpt.train --config configs/kaggle_t4.json

## Ablations (2,000 iterations each)
One change at a time versus the baseline, all with the same seed and data.

In [ ]:
ablations = {
    "abl_base": "",
    "abl_depth4": "--n_layer 4",
    "abl_depth8": "--n_layer 8",
    "abl_ctx128": "--block_size 128 --batch_size 128",  # same tokens per step
    "abl_lr3e-3": "--lr 0.003 --min_lr 0.0003",
}
for name, args in ablations.items():
    !python -m forgegpt.train --config configs/kaggle_t4.json --out_dir /kaggle/working/runs/{name} --max_iters 2000 --eval_interval 250 {args}

In [ ]:
!python scripts/summarize_runs.py /kaggle/working/runs --out /kaggle/working/ablations.md
print(open("/kaggle/working/ablations.md").read())

## Loss curves and samples

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, glob, os

fig, ax = plt.subplots(figsize=(8, 4.5))
for path in sorted(glob.glob("/kaggle/working/runs/*/eval.csv")):
    df = pd.read_csv(path)
    ax.plot(df["tokens_seen"] / 1e6, df["val_loss"], label=path.split("/")[-2])
ax.set_xlabel("training tokens (millions)")
ax.set_ylabel("validation loss")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print(open("/kaggle/working/runs/t4_6L384/samples.txt").read()[-3000:])

## Export
Download `runs/t4_6L384/ckpt_best.pt` (Output tab), put it in `runs/t4_6L384/` locally, then:

```bash
python scripts/benchmark_inference.py --ckpt runs/t4_6L384/ckpt_best.pt
python scripts/export_onnx.py --ckpt runs/t4_6L384/ckpt_best.pt
```